In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

In [3]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

--2026-07-09 19:21:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-07-09 19:21:33 (15.2 MB/s) - ‘rag_helper.py’ saved [2134/2134]

--2026-07-09 19:21:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response.

In [7]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [13]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [14]:
import json
user_prompt = json.dumps(doc)

In [11]:
from pydantic import BaseModel
from evaluation_utils import llm_structured, llm_structured_retry

class Questions(BaseModel):
    questions: list[str]

def generate_ground_truth(doc):
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"],
    })

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )
    records = []
    for q in out.questions:
        records.append({
            "question": q,
            "filename": doc["filename"],
        })

    return records, usage

In [15]:
first_three = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

docs_by_filename = {doc["filename"]: doc for doc in documents}

usages = []
for filename in first_three:
    doc = docs_by_filename[filename]
    user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})
    _, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)
    usages.append(usage)
    print(filename, usage.input_tokens)

avg_input_tokens = sum(u.input_tokens for u in usages) / len(usages)
print("Q1 average input tokens:", avg_input_tokens)

01-agentic-rag/lessons/01-intro.md 1021
01-agentic-rag/lessons/02-environment.md 1287
01-agentic-rag/lessons/03-rag.md 1754
Q1 average input tokens: 1354.0


In [17]:
import pandas as pd

In [18]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget -O ground-truth.csv {PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)  # 360

--2026-07-09 19:40:58--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

ground-truth.csv    100%[===================>]  47.49K  --.-KB/s    in 0.07s   

2026-07-09 19:41:01 (659 KB/s) - ‘ground-truth.csv’ saved [48627/48627]



360

In [20]:
from gitsource import chunk_documents
from minsearch import Index, VectorSearch

In [21]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [23]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [24]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [25]:
q = ground_truth[0]["question"]
print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [26]:
text_results = text_search(q)
text_results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

In [32]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"

!wget -nc {PREFIX}/embedder.py
!wget -nc {PREFIX}/download.py
!python download.py

File ‘embedder.py’ already there; not retrieving.

--2026-07-09 21:36:04--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

download.py         100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-07-09 21:36:05 (36.4 MB/s) - ‘download.py’ saved [1376/1376]

  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


In [33]:
from embedder import Embedder

embed = Embedder()

In [34]:
texts = [chunk["content"] for chunk in chunks]
X = embed.encode_batch(texts)

In [35]:
X.shape

(295, 384)

In [36]:
vector_index = VectorSearch()
vector_index.fit(X, chunks)

In [37]:
def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    return vector_index.search(query_vector, num_results=num_results)

In [38]:
vector_results = vector_search(q)
vector_results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [39]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [40]:
def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

In [41]:
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [42]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance(q, search_function))
    return relevance_total

In [43]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)

In [44]:
def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance)

In [45]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [46]:
result_text = evaluate(ground_truth, text_search)
result_text

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [47]:
result_vector = evaluate(ground_truth, vector_search)
result_vector

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [48]:
results_by_k = {}
for k in [1, 50, 100, 200]:
    result = evaluate(ground_truth, lambda query, k=k: hybrid_search(query, k=k))
    results_by_k[k] = result
    print(f"k={k}: {result}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


In [49]:
best_k = max(results_by_k, key=lambda k: (results_by_k[k]["mrr"], -k))
best_k

1